# 113. Voronoi分割: 画像 vs テキストの最適パラメータ比較

## 目的
- 画像Embedding（顔認識, 75K, 512D）ではC=256, P=2で有効だったVoronoi分割が、
  テキストEmbedding（E5-base, 10K, 768D）ではなぜ異なるパラメータが必要かを定量的に分析
- **候補割合（候補数/N）を揃えた**公平な比較により、最適な(C, P)を特定

## 画像プロジェクトの参考値
| 構成 | N | 候補数 | 候補割合 | BF-R@30 |
|------|---|--------|---------|---------|
| C=128, P=5 | 75,162 | ~3,200 | 4.3% | 83.3% |
| C=256, P=2 (assign=2) | 75,162 | ~3,200 | 4.3% | 83.3% |

## 比較の軸
1. **候補割合を固定**（2%, 5%, 10%, 20%）して、C × Pの最適組み合わせを探索
2. **パーティション品質**の比較（クラスタのコンパクトさ、分離度）
3. **分布の一様性**: テキスト vs 画像でembedding空間の構造がどう違うか

## 0. セットアップ

In [1]:
import sys
import numpy as np
import time
from pathlib import Path
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '../src')

DATA_DIR = Path('../data')
np.random.seed(42)

N_QUERIES = 200
TOP_K = 10
print(f'Configuration: N_QUERIES={N_QUERIES}, TOP_K={TOP_K}')

Configuration: N_QUERIES=200, TOP_K=10


## 1. データロード

In [2]:
# テキストEmbedding
emb_en = np.load(DATA_DIR / '10k_e5_base_en_embeddings.npy')
emb_ja = np.load(DATA_DIR / '10k_e5_base_ja_embeddings.npy')

datasets = {
    'Text-EN': emb_en,
    'Text-JA': emb_ja,
}

for name, emb in datasets.items():
    norms = np.linalg.norm(emb, axis=1)
    print(f'{name}: shape={emb.shape}, norm: mean={norms.mean():.4f}, std={norms.std():.4f}')

Text-EN: shape=(10000, 768), norm: mean=1.0000, std=0.0000
Text-JA: shape=(9990, 768), norm: mean=1.0000, std=0.0000


## 2. Embedding空間の構造分析

テキストEmbeddingの分布特性を分析し、画像Embeddingとの違いを理解する。
- ペア間cosine類似度の分布: 一様なら類似度が集中、クラスタ構造があれば分散
- 次元の有効ランク: 情報が集中しているか拡散しているか

In [3]:
def analyze_embedding_space(embeddings, name, n_sample=5000):
    """Embedding空間の構造を分析"""
    N, D = embeddings.shape
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    emb_normed = embeddings / norms
    
    # ランダムペアのcosine類似度分布
    rng = np.random.default_rng(42)
    idx1 = rng.choice(N, n_sample, replace=True)
    idx2 = rng.choice(N, n_sample, replace=True)
    mask = idx1 != idx2
    idx1, idx2 = idx1[mask], idx2[mask]
    cos_sims = np.sum(emb_normed[idx1] * emb_normed[idx2], axis=1)
    
    # k近傍のcosine類似度（top-10の平均）
    query_ids = rng.choice(N, min(200, N), replace=False)
    knn_sims = []
    for qi in query_ids:
        sims = emb_normed[qi] @ emb_normed.T
        sims[qi] = -1
        top10_sims = np.sort(sims)[-10:]
        knn_sims.append(np.mean(top10_sims))
    
    # PCA分散（有効次元数の指標）
    centered = emb_normed - emb_normed.mean(axis=0)
    cov = centered.T @ centered / N
    eigvals = np.linalg.eigvalsh(cov)[::-1]
    eigvals_norm = eigvals / eigvals.sum()
    cum_var = np.cumsum(eigvals_norm)
    eff_dim_90 = np.searchsorted(cum_var, 0.90) + 1
    eff_dim_95 = np.searchsorted(cum_var, 0.95) + 1
    
    print(f'\n{"="*60}')
    print(f'{name} (N={N}, D={D})')
    print(f'{"="*60}')
    print(f'  ランダムペア cosine:  mean={cos_sims.mean():.4f}, std={cos_sims.std():.4f}')
    print(f'                        min={cos_sims.min():.4f}, max={cos_sims.max():.4f}')
    print(f'  k=10近傍 cosine:      mean={np.mean(knn_sims):.4f}, std={np.std(knn_sims):.4f}')
    print(f'  ランダム-近傍 Gap:    {np.mean(knn_sims) - cos_sims.mean():.4f}')
    print(f'  有効次元数 (90%分散): {eff_dim_90}')
    print(f'  有効次元数 (95%分散): {eff_dim_95}')
    print(f'  Top-10固有値占有率:   {cum_var[9]:.4f}')
    
    return {
        'name': name, 'N': N, 'D': D,
        'cos_mean': cos_sims.mean(), 'cos_std': cos_sims.std(),
        'knn_mean': np.mean(knn_sims), 'knn_std': np.std(knn_sims),
        'gap': np.mean(knn_sims) - cos_sims.mean(),
        'eff_dim_90': eff_dim_90, 'eff_dim_95': eff_dim_95,
        'top10_var': cum_var[9],
    }


# 画像プロジェクトの参考値（NB100より）
print('='*60)
print('画像プロジェクト参考値（ArcFace 512D, 75K顔画像）')
print('='*60)
print('  ランダムペア cosine:  mean≈0.05, std≈0.08')
print('  k=10近傍 cosine:      mean≈0.55〜0.70 (同一人物クラスタ)')
print('  ランダム-近傍 Gap:    ≈0.50〜0.65 (非常に大きい)')
print('  → 顔画像は強いクラスタ構造を持つ')

# テキスト分析
text_stats = {}
for name, emb in datasets.items():
    text_stats[name] = analyze_embedding_space(emb, name)

print(f'\n{"="*60}')
print('画像 vs テキストの構造比較')
print(f'{"="*60}')
print(f'{"指標":<25} {"画像(参考)":>12} {"Text-EN":>12} {"Text-JA":>12}')
print('-' * 65)
print(f'{"ランダムペアcos mean":<25} {"~0.05":>12} '
      f'{text_stats["Text-EN"]["cos_mean"]:>12.4f} '
      f'{text_stats["Text-JA"]["cos_mean"]:>12.4f}')
print(f'{"近傍cos mean":<25} {"~0.60":>12} '
      f'{text_stats["Text-EN"]["knn_mean"]:>12.4f} '
      f'{text_stats["Text-JA"]["knn_mean"]:>12.4f}')
print(f'{"Gap (近傍-ランダム)":<25} {"~0.55":>12} '
      f'{text_stats["Text-EN"]["gap"]:>12.4f} '
      f'{text_stats["Text-JA"]["gap"]:>12.4f}')
print(f'{"有効次元(90%)":<25} {"~50":>12} '
      f'{text_stats["Text-EN"]["eff_dim_90"]:>12} '
      f'{text_stats["Text-JA"]["eff_dim_90"]:>12}')

画像プロジェクト参考値（ArcFace 512D, 75K顔画像）
  ランダムペア cosine:  mean≈0.05, std≈0.08
  k=10近傍 cosine:      mean≈0.55〜0.70 (同一人物クラスタ)
  ランダム-近傍 Gap:    ≈0.50〜0.65 (非常に大きい)
  → 顔画像は強いクラスタ構造を持つ



Text-EN (N=10000, D=768)
  ランダムペア cosine:  mean=0.7063, std=0.0241
                        min=0.6232, max=0.8465
  k=10近傍 cosine:      mean=0.8150, std=0.0270
  ランダム-近傍 Gap:    0.1088
  有効次元数 (90%分散): 313
  有効次元数 (95%分散): 386
  Top-10固有値占有率:   0.1521

Text-JA (N=9990, D=768)
  ランダムペア cosine:  mean=0.7651, std=0.0293
                        min=0.6645, max=0.9447
  k=10近傍 cosine:      mean=0.8730, std=0.0270
  ランダム-近傍 Gap:    0.1080
  有効次元数 (90%分散): 295
  有効次元数 (95%分散): 372
  Top-10固有値占有率:   0.2432

画像 vs テキストの構造比較
指標                              画像(参考)      Text-EN      Text-JA
-----------------------------------------------------------------
ランダムペアcos mean                   ~0.05       0.7063       0.7651
近傍cos mean                       ~0.60       0.8150       0.8730
Gap (近傍-ランダム)                    ~0.55       0.1088       0.1080
有効次元(90%)                          ~50          313          295


## 3. 候補割合を固定した網羅的グリッドサーチ

画像プロジェクトと公平に比較するため、**候補割合（候補数/N）**を固定して
最適な(C, P)の組み合わせを探索する。

画像プロジェクト: C=128/256, P=2〜5で候補割合4.3%
→ テキストでは候補割合2%, 5%, 10%, 20%の各帯で最適構成を特定

In [4]:
def get_ground_truth(embeddings, qi, top_k=10):
    cos_sims = cosine_similarity(embeddings[qi:qi+1], embeddings)[0]
    cos_sims[qi] = -1
    return set(np.argsort(cos_sims)[-top_k:])


def build_voronoi_model(embeddings, n_clusters):
    """k-meansを1回だけ実行してモデルを構築"""
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    emb_normed = embeddings / norms
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=2048, n_init=3)
    labels = kmeans.fit_predict(emb_normed)
    centroids = kmeans.cluster_centers_
    centroids_normed = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)
    partitions = {c: np.where(labels == c)[0] for c in range(n_clusters)}
    sizes = [len(v) for v in partitions.values()]
    return centroids_normed, labels, partitions, sizes


def evaluate_voronoi(embeddings, emb_normed, centroids, partitions, 
                     n_probes, n_queries=200, top_k=10, seed=42):
    """構築済みモデルで評価（高速）"""
    N = len(embeddings)
    rng = np.random.default_rng(seed)
    query_indices = rng.choice(N, min(n_queries, N // 2), replace=False)
    
    recalls = []
    candidate_counts = []
    
    for qi in query_indices:
        gt = get_ground_truth(embeddings, qi, top_k)
        sims = centroids @ emb_normed[qi]
        top_c = np.argsort(-sims)[:n_probes]
        candidates = np.concatenate([partitions[c] for c in top_c])
        candidates = candidates[candidates != qi]
        candidate_counts.append(len(candidates))
        
        if len(candidates) > 0:
            cand_sims = cosine_similarity(embeddings[qi:qi+1], embeddings[candidates])[0]
            top_in_cand = candidates[np.argsort(-cand_sims)[:top_k]]
            recalls.append(len(gt & set(top_in_cand)) / top_k)
        else:
            recalls.append(0.0)
    
    return np.mean(recalls), np.mean(candidate_counts)


# 事前にk-meansモデルを全構築
cluster_range = [8, 16, 32, 64, 128, 256]
probe_range = [1, 2, 3, 4, 5, 8, 10, 15, 20]

voronoi_cache = {}
for ds_name, emb in datasets.items():
    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    emb_normed = emb / norms
    voronoi_cache[ds_name] = {'emb_normed': emb_normed}
    print(f'\n{ds_name}: k-means構築中...')
    for n_c in cluster_range:
        centroids, labels, partitions, sizes = build_voronoi_model(emb, n_c)
        voronoi_cache[ds_name][n_c] = {
            'centroids': centroids, 'labels': labels,
            'partitions': partitions, 'sizes': sizes,
        }
        print(f'  C={n_c}: mean={np.mean(sizes):.1f}, min={np.min(sizes)}, '
              f'max={np.max(sizes)}, CV={np.std(sizes)/np.mean(sizes):.3f}')

# グリッドサーチ（k-meansはキャッシュ済み）
all_grid = {}
for ds_name, emb in datasets.items():
    print(f'\n{"="*80}')
    print(f'{ds_name} (N={len(emb)}) グリッドサーチ')
    print(f'{"="*80}')
    
    emb_normed = voronoi_cache[ds_name]['emb_normed']
    results = []
    for n_c in cluster_range:
        model = voronoi_cache[ds_name][n_c]
        for n_p in probe_range:
            if n_p > n_c:
                continue
            recall, cands = evaluate_voronoi(
                emb, emb_normed, model['centroids'], model['partitions'],
                n_probes=n_p, n_queries=N_QUERIES
            )
            results.append({
                'n_clusters': n_c, 'n_probes': n_p,
                'recall': recall, 'candidates': cands,
                'cand_ratio': cands / len(emb),
                'partition_cv': np.std(model['sizes']) / np.mean(model['sizes']),
            })
    
    all_grid[ds_name] = results
    
    # 候補割合帯ごとの最適構成
    print(f'\n--- 候補割合帯ごとの最適構成 ---')
    target_ratios = [(0.01, 0.03, '~2%'), (0.03, 0.07, '~5%'), 
                     (0.07, 0.15, '~10%'), (0.15, 0.25, '~20%'),
                     (0.25, 0.40, '~30%')]
    
    print(f'{"帯":<8} {"Best (C,P)":<15} {"R@10":>8} {"候補数":>8} {"候補%":>8} {"CV":>8}')
    print('-' * 60)
    for lo, hi, label in target_ratios:
        in_band = [r for r in results if lo <= r['cand_ratio'] < hi]
        if not in_band:
            continue
        best = max(in_band, key=lambda x: x['recall'])
        config = f'C={best["n_clusters"]},P={best["n_probes"]}'
        print(f'{label:<8} {config:<15} {best["recall"]*100:>7.1f}% '
              f'{best["candidates"]:>7.0f} {best["cand_ratio"]*100:>7.1f}% '
              f'{best["partition_cv"]:>7.3f}')


Text-EN: k-means構築中...


  C=8: mean=1250.0, min=857, max=1638, CV=0.255


  C=16: mean=625.0, min=287, max=983, CV=0.293


  C=32: mean=312.5, min=105, max=526, CV=0.382


  C=64: mean=156.2, min=43, max=266, CV=0.381


...

  C=64: mean=156.1, min=50, max=356, CV=0.457


  C=128: mean=78.0, min=1, max=405, CV=0.798


  C=256: mean=39.0, min=1, max=244, CV=0.959

Text-EN (N=10000) グリッドサーチ



--- 候補割合帯ごとの最適構成 ---
帯        Best (C,P)          R@10      候補数      候補%       CV
------------------------------------------------------------
~2%      C=256,P=4          77.8%     277     2.8%   0.858
~5%      C=256,P=10         89.1%     676     6.8%   0.858
~10%     C=256,P=20         93.5%    1333    13.3%   0.858
~20%     C=128,P=20         94.6%    1841    18.4%   0.556
~30%     C=64,P=20          97.1%    3486    34.9%   0.381

Text-JA (N=9990) グリッドサーチ



--- 候補割合帯ごとの最適構成 ---
帯        Best (C,P)          R@10      候補数      候補%       CV
------------------------------------------------------------
~2%      C=256,P=4          85.2%     275     2.8%   0.959
~5%      C=256,P=10         93.7%     641     6.4%   0.959
~10%     C=128,P=15         97.8%    1380    13.8%   0.798
~20%     C=64,P=15          98.7%    2455    24.6%   0.457
~30%     C=64,P=20          99.3%    3243    32.5%   0.457


## 4. 同一候補割合でのC × P全組み合わせヒートマップ

候補割合~5%（画像プロジェクトの4.3%に相当）付近で、
CとPのどの組み合わせが最もRecallが高いかを可視化する。

In [5]:
print('='*80)
print('C × P 全結果テーブル（R@10 / 候補割合）')
print('='*80)

for ds_name in ['Text-EN', 'Text-JA']:
    results = all_grid[ds_name]
    
    print(f'\n--- {ds_name} ---')
    
    # Cごとにまとめる
    c_values = sorted(set(r['n_clusters'] for r in results))
    p_values = sorted(set(r['n_probes'] for r in results))
    
    # R@10テーブル
    header = f'{"C \\ P":<8}' + ''.join(f'{p:>8}' for p in p_values)
    print(f'\nR@10 (%):\n{header}')
    print('-' * (8 + 8 * len(p_values)))
    for c in c_values:
        row = f'{c:<8}'
        for p in p_values:
            match = [r for r in results if r['n_clusters'] == c and r['n_probes'] == p]
            if match:
                row += f'{match[0]["recall"]*100:>7.1f}%'
            else:
                row += f'{"---":>8}'
        print(row)
    
    # 候補割合テーブル
    print(f'\n候補割合 (%):\n{header}')
    print('-' * (8 + 8 * len(p_values)))
    for c in c_values:
        row = f'{c:<8}'
        for p in p_values:
            match = [r for r in results if r['n_clusters'] == c and r['n_probes'] == p]
            if match:
                row += f'{match[0]["cand_ratio"]*100:>7.1f}%'
            else:
                row += f'{"---":>8}'
        print(row)

C × P 全結果テーブル（R@10 / 候補割合）

--- Text-EN ---

R@10 (%):
C \ P          1       2       3       4       5       8      10      15      20
--------------------------------------------------------------------------------
8          76.0%   88.4%   94.2%   96.8%   97.9%  100.0%     ---     ---     ---
16         68.2%   82.5%   88.2%   93.2%   95.2%   98.2%   99.2%  100.0%     ---
32         64.0%   78.4%   84.2%   88.2%   90.9%   95.1%   97.0%   98.8%   99.7%
64         60.9%   74.6%   82.1%   85.6%   88.2%   91.8%   93.7%   96.4%   97.1%
128        57.2%   68.8%   75.2%   79.5%   82.2%   87.4%   89.7%   93.0%   94.6%
256        54.2%   67.4%   73.9%   77.8%   81.0%   86.4%   89.1%   92.0%   93.5%

候補割合 (%):
C \ P          1       2       3       4       5       8      10      15      20
--------------------------------------------------------------------------------
8          13.7%   26.1%   38.4%   50.1%   62.0%  100.0%     ---     ---     ---
16          6.9%   13.6%   20.2%   26.8%   

## 5. 画像条件の再現: C=256, P=2 と同等候補数での比較

画像プロジェクトのC=256, P=2（候補割合4.3%）をテキストで再現し、
同じ候補割合で最適なテキスト用構成と比較する。

In [6]:
print('='*80)
print('画像条件の再現テスト')
print('='*80)

# 画像: C=256, P=2, 候補割合4.3%, R@30≈83%
print('\n--- 画像と同じC=256, P=2 ---')
for ds_name in ['Text-EN', 'Text-JA']:
    r256p2 = [r for r in all_grid[ds_name] 
              if r['n_clusters'] == 256 and r['n_probes'] == 2]
    if r256p2:
        r = r256p2[0]
        print(f'  {ds_name}: R@10={r["recall"]*100:.1f}%, '
              f'候補={r["candidates"]:.0f} ({r["cand_ratio"]*100:.1f}%), '
              f'CV={r["partition_cv"]:.3f}')

# 同等候補割合（3-7%）で最適な構成
print('\n--- 候補割合3-7%（画像の4.3%に相当）での最適構成 ---')
for ds_name in ['Text-EN', 'Text-JA']:
    in_band = [r for r in all_grid[ds_name] if 0.03 <= r['cand_ratio'] < 0.07]
    if in_band:
        best = max(in_band, key=lambda x: x['recall'])
        worst = min(in_band, key=lambda x: x['recall'])
        
        print(f'\n  {ds_name}:')
        print(f'    Best:  C={best["n_clusters"]},P={best["n_probes"]} → '
              f'R@10={best["recall"]*100:.1f}%, 候補={best["candidates"]:.0f} ({best["cand_ratio"]*100:.1f}%)')
        print(f'    Worst: C={worst["n_clusters"]},P={worst["n_probes"]} → '
              f'R@10={worst["recall"]*100:.1f}%, 候補={worst["candidates"]:.0f} ({worst["cand_ratio"]*100:.1f}%)')
        
        sorted_band = sorted(in_band, key=lambda x: x['recall'], reverse=True)
        print(f'    全構成 ({len(sorted_band)}件):')
        for r in sorted_band:
            print(f'      C={r["n_clusters"]:>3},P={r["n_probes"]:>2}: '
                  f'R@10={r["recall"]*100:>5.1f}%, 候補={r["candidates"]:>5.0f} ({r["cand_ratio"]*100:.1f}%)')

# 画像との直接比較
print('\n--- 画像 vs テキスト 直接比較 ---')
print(f'{"条件":<25} {"R@10/R@30":>10} {"候補割合":>10} {"構成":>15}')
print('-' * 65)
print(f'{"画像(顔認識)":<25} {"83.3%":>10} {"4.3%":>10} {"C=256,P=2":>15}')
for ds_name in ['Text-EN', 'Text-JA']:
    r = [r for r in all_grid[ds_name] if r['n_clusters'] == 256 and r['n_probes'] == 2]
    if r:
        print(f'{ds_name + "(C=256,P=2)":<25} {r[0]["recall"]*100:>9.1f}% '
              f'{r[0]["cand_ratio"]*100:>9.1f}% {"C=256,P=2":>15}')
    best = max([r for r in all_grid[ds_name] if 0.03 <= r['cand_ratio'] < 0.07],
               key=lambda x: x['recall'], default=None)
    if best:
        config = f'C={best["n_clusters"]},P={best["n_probes"]}'
        print(f'{ds_name + "(最適)":<25} {best["recall"]*100:>9.1f}% '
              f'{best["cand_ratio"]*100:>9.1f}% {config:>15}')

画像条件の再現テスト

--- 画像と同じC=256, P=2 ---
  Text-EN: R@10=67.4%, 候補=140 (1.4%), CV=0.858
  Text-JA: R@10=73.4%, 候補=145 (1.5%), CV=0.959

--- 候補割合3-7%（画像の4.3%に相当）での最適構成 ---

  Text-EN:
    Best:  C=256,P=10 → R@10=89.1%, 候補=676 (6.8%)
    Worst: C=32,P=1 → R@10=64.0%, 候補=361 (3.6%)
    全構成 (9件):
      C=256,P=10: R@10= 89.1%, 候補=  676 (6.8%)
      C=256,P= 8: R@10= 86.4%, 候補=  544 (5.4%)
      C=128,P= 5: R@10= 82.2%, 候補=  470 (4.7%)
      C= 64,P= 3: R@10= 82.1%, 候補=  549 (5.5%)
      C=256,P= 5: R@10= 81.0%, 候補=  344 (3.4%)
      C=128,P= 4: R@10= 79.5%, 候補=  379 (3.8%)
      C= 64,P= 2: R@10= 74.6%, 候補=  365 (3.6%)
      C= 16,P= 1: R@10= 68.2%, 候補=  687 (6.9%)
      C= 32,P= 1: R@10= 64.0%, 候補=  361 (3.6%)

  Text-JA:
    Best:  C=256,P=10 → R@10=93.7%, 候補=641 (6.4%)
    Worst: C=32,P=1 → R@10=71.2%, 候補=353 (3.5%)
    全構成 (12件):
      C=256,P=10: R@10= 93.7%, 候補=  641 (6.4%)
      C=256,P= 8: R@10= 91.9%, 候補=  519 (5.2%)
      C= 64,P= 4: R@10= 91.1%, 候補=  695 (7.0%)
      C=128,P= 5: R@1

## 6. パーティション品質分析

Cの値によってパーティションの「コンパクトさ」がどう変わるかを分析。
パーティション内のcosine類似度が高いほど、少ないprobeで近傍を捕捉できる。

In [7]:
def analyze_partition_quality(emb_normed, labels, n_clusters, n_sample_pairs=5000):
    """パーティションの品質を分析（キャッシュ済みモデルを使用）"""
    N = len(emb_normed)
    rng = np.random.default_rng(42)
    
    # パーティション内cosine類似度
    intra_sims = []
    for c in range(n_clusters):
        members = np.where(labels == c)[0]
        if len(members) < 2:
            continue
        sample = rng.choice(members, min(20, len(members)), replace=False)
        for i in range(len(sample)):
            for j in range(i + 1, len(sample)):
                intra_sims.append(emb_normed[sample[i]] @ emb_normed[sample[j]])
    
    # パーティション間cosine類似度
    inter_sims = []
    for _ in range(n_sample_pairs):
        c1, c2 = rng.choice(n_clusters, 2, replace=False)
        m1 = np.where(labels == c1)[0]
        m2 = np.where(labels == c2)[0]
        if len(m1) == 0 or len(m2) == 0:
            continue
        inter_sims.append(emb_normed[rng.choice(m1)] @ emb_normed[rng.choice(m2)])
    
    # top-10近傍がいくつのパーティションに散らばるか
    query_ids = rng.choice(N, min(200, N), replace=False)
    n_partitions_for_top10 = []
    for qi in query_ids:
        sims = emb_normed[qi] @ emb_normed.T
        sims[qi] = -1
        top10 = np.argsort(-sims)[:10]
        n_partitions_for_top10.append(len(set(labels[top10])))
    
    return {
        'n_clusters': n_clusters,
        'intra_cos_mean': np.mean(intra_sims),
        'inter_cos_mean': np.mean(inter_sims),
        'separation': np.mean(intra_sims) - np.mean(inter_sims),
        'top10_partitions_mean': np.mean(n_partitions_for_top10),
        'top10_partitions_std': np.std(n_partitions_for_top10),
        'top10_partitions_max': np.max(n_partitions_for_top10),
    }


print('='*80)
print('パーティション品質分析')
print('='*80)

for ds_name in ['Text-EN', 'Text-JA']:
    emb_normed = voronoi_cache[ds_name]['emb_normed']
    
    print(f'\n--- {ds_name} ---')
    print(f'{"C":<6} {"内cos":>8} {"間cos":>8} {"分離":>8} '
          f'{"top10散布":>10} {"std":>6} {"max":>5}')
    print('-' * 55)
    
    for n_c in cluster_range:
        model = voronoi_cache[ds_name][n_c]
        q = analyze_partition_quality(emb_normed, model['labels'], n_c)
        print(f'{n_c:<6} {q["intra_cos_mean"]:>7.4f} {q["inter_cos_mean"]:>7.4f} '
              f'{q["separation"]:>7.4f} '
              f'{q["top10_partitions_mean"]:>9.1f} {q["top10_partitions_std"]:>5.1f} '
              f'{q["top10_partitions_max"]:>4}')

print('\n※ top10散布 = top-10近傍が何個のパーティションに分散するか')
print('  この値がn_probesより大きいと取りこぼしが発生')
print('  画像(顔認識): 同一人物の顔が1クラスタに集中 → 散布が小さい')
print('  テキスト(Wikipedia): 多様なトピック → 散布が大きい')

パーティション品質分析

--- Text-EN ---
C          内cos     間cos       分離    top10散布    std   max
-------------------------------------------------------


8       0.7337  0.7031  0.0305       2.0   1.1    6


16      0.7389  0.7049  0.0339       2.6   1.4    9


32      0.7496  0.7050  0.0446       2.9   1.5    7


64      0.7583  0.7048  0.0535       3.3   1.9    8


...

16      0.8022  0.7604  0.0417       2.3   1.4    8


32      0.8181  0.7625  0.0556       2.4   1.4    7


64      0.8227  0.7627  0.0600       2.6   1.4    7


128     0.8276  0.7605  0.0671       3.0   1.7    9


256     0.8363  0.7613  0.0750       3.6   1.7    9

※ top10散布 = top-10近傍が何個のパーティションに分散するか
  この値がn_probesより大きいと取りこぼしが発生
  画像(顔認識): 同一人物の顔が1クラスタに集中 → 散布が小さい
  テキスト(Wikipedia): 多様なトピック → 散布が大きい


## 7. テキスト向け推奨パラメータの導出

上記分析を踏まえ、テキストEmbeddingにおけるVoronoi分割の推奨パラメータをまとめる。

In [8]:
print('='*80)
print('テキスト向け推奨パラメータ')
print('='*80)

print('''
【画像 vs テキストの根本的な違い】

画像(顔認識):
  - 強いクラスタ構造（同一人物、人種、年齢でグループ化）
  - ランダム-近傍Gap ≈ 0.55 → 近傍と非近傍が明確に分離
  - C=256で十分に分離された小パーティションが作れる
  - P=2で近傍のほとんどをカバー可能

テキスト(Wikipedia E5-base):
  - 弱いクラスタ構造（トピック間の境界が曖昧）
  - ランダム-近傍Gap が画像より小さい → 近傍が複数パーティションに散布
  - top-10近傍の散布先パーティション数が多い → probeを増やす必要あり
  - 少ないCで粗い分割にし、probeで隣接をカバーする方が効率的
''')

# 推奨構成の提示
print('【推奨構成（10Kスケール）】')
print(f'\n{"用途":<20} {"構成":>15} {"R@10":>8} {"候補割合":>10} {"Firestore IN句":>15}')
print('-' * 72)

for ds_name in ['Text-EN', 'Text-JA']:
    results = all_grid[ds_name]
    print(f'\n  [{ds_name}]')
    
    # R@10 >= 90%の最小候補数
    good90 = sorted([r for r in results if r['recall'] >= 0.90], key=lambda x: x['candidates'])
    if good90:
        r = good90[0]
        print(f'  {"R@10≥90%最小候補":<18} C={r["n_clusters"]},P={r["n_probes"]:>13} '
              f'{r["recall"]*100:>7.1f}% {r["cand_ratio"]*100:>9.1f}% {r["n_probes"]:>14}')
    
    # R@10 >= 85%の最小候補数
    good85 = sorted([r for r in results if r['recall'] >= 0.85], key=lambda x: x['candidates'])
    if good85:
        r = good85[0]
        print(f'  {"R@10≥85%最小候補":<18} C={r["n_clusters"]},P={r["n_probes"]:>13} '
              f'{r["recall"]*100:>7.1f}% {r["cand_ratio"]*100:>9.1f}% {r["n_probes"]:>14}')
    
    # R@10 >= 80%の最小候補数
    good80 = sorted([r for r in results if r['recall'] >= 0.80], key=lambda x: x['candidates'])
    if good80:
        r = good80[0]
        print(f'  {"R@10≥80%最小候補":<18} C={r["n_clusters"]},P={r["n_probes"]:>13} '
              f'{r["recall"]*100:>7.1f}% {r["cand_ratio"]*100:>9.1f}% {r["n_probes"]:>14}')

テキスト向け推奨パラメータ

【画像 vs テキストの根本的な違い】

画像(顔認識):
  - 強いクラスタ構造（同一人物、人種、年齢でグループ化）
  - ランダム-近傍Gap ≈ 0.55 → 近傍と非近傍が明確に分離
  - C=256で十分に分離された小パーティションが作れる
  - P=2で近傍のほとんどをカバー可能

テキスト(Wikipedia E5-base):
  - 弱いクラスタ構造（トピック間の境界が曖昧）
  - ランダム-近傍Gap が画像より小さい → 近傍が複数パーティションに散布
  - top-10近傍の散布先パーティション数が多い → probeを増やす必要あり
  - 少ないCで粗い分割にし、probeで隣接をカバーする方が効率的

【推奨構成（10Kスケール）】

用途                                構成     R@10       候補割合   Firestore IN句
------------------------------------------------------------------------

  [Text-EN]
  R@10≥90%最小候補       C=256,P=           15    92.0%      10.1%             15
  R@10≥85%最小候補       C=256,P=            8    86.4%       5.4%              8
  R@10≥80%最小候補       C=256,P=            5    81.0%       3.4%              5

  [Text-JA]
  R@10≥90%最小候補       C=256,P=            8    91.9%       5.2%              8
  R@10≥85%最小候補       C=256,P=            4    85.2%       2.8%              4
  R@10≥80%最小候補       C=256,P=            3    80.7%       2.1%              3


## 8. 評価・考察

### 画像とテキストでVoronoiの最適パラメータが異なる根本原因

**ランダム-近傍Gap（近傍とそれ以外のcosine類似度の差）が決定的に違う:**

| 指標 | 画像(顔認識) | Text-EN | Text-JA |
|------|------------|---------|---------|
| ランダムペア cos | ~0.05 | 0.706 | 0.765 |
| 近傍 cos (top-10) | ~0.60 | 0.815 | 0.873 |
| **Gap** | **~0.55** | **0.109** | **0.108** |

画像のGap(0.55)はテキスト(0.11)の**5倍**。これは顔画像に同一人物・人種等の自然なクラスタ構造があるのに対し、テキスト(Wikipedia)はトピックが高次元空間に一様に分散しているため。

### top-10近傍の散布パーティション数が核心

| C | Text-EN散布 | Text-JA散布 |
|---|------------|------------|
| 8 | 2.0 | 1.7 |
| 32 | 2.9 | 2.4 |
| 64 | 3.3 | 2.6 |
| 128 | 3.8 | 3.0 |
| 256 | 4.0 | 3.6 |

C=256ではtop-10近傍が**平均4パーティション（最大9）**に散らばる。P=2ではそのうち2つしかカバーできないため、R@10=67-73%に留まる。画像では同一人物クラスタが1パーティションに収まるためP=2で十分。

### 画像条件(C=256, P=2)をテキストに適用した結果

| 条件 | R@10 | 候補割合 |
|------|------|---------|
| 画像 C=256,P=2 | 83.3% | 4.3% |
| Text-EN C=256,P=2 | **67.4%** | 1.4% |
| Text-JA C=256,P=2 | **73.4%** | 1.5% |

テキストでは候補割合も1.4%と画像の4.3%より大幅に少ない（10Kを256分割 → 1パーティション39件、画像は75Kを256分割 → 294件）。

### 候補割合を揃えた公平比較（3-7%帯）

| 条件 | R@10 | 候補割合 | 構成 |
|------|------|---------|------|
| 画像 | 83.3% | 4.3% | C=256,P=2 |
| Text-EN (最適) | **89.1%** | 6.8% | **C=256,P=10** |
| Text-JA (最適) | **93.7%** | 6.4% | **C=256,P=10** |

候補割合を揃えると、テキストの方が**画像より高いRecall**を達成。テキストはGapが小さいが、Voronoiで十分な候補を集めればcosine rerankで正確に選別できる。

### テキスト向け推奨パラメータ

**C=256を維持し、P(probe数)を増やす**のが最適解:

| 目標 | Text-EN | Text-JA |
|------|---------|---------|
| R@10≥90% | C=256, P=15 (候補10.1%) | C=256, P=8 (候補5.2%) |
| R@10≥85% | C=256, P=8 (候補5.4%) | C=256, P=4 (候補2.8%) |
| R@10≥80% | C=256, P=5 (候補3.4%) | C=256, P=3 (候補2.1%) |

Firestore IN句は最大30要素まで対応しているため、P=15でも十分に運用可能。

### 画像のC=256, P=2に対応するテキスト構成

画像と同等のRecall(~83%)を達成するテキスト構成:
- **Text-EN: C=256, P=5** → R@10=81.0%, 候補3.4%
- **Text-JA: C=256, P=3** → R@10=80.7%, 候補2.1%

つまり**画像のP=2に対してテキストはP=3〜5**が同等で、「桁違い」ではなく2〜3倍のprobeで済む。

### 結論

1. **Cは画像と同じ256で問題ない**。パーティション数を減らす必要はない
2. **Pを画像の2〜5倍（P=5〜10）にすれば、画像と同等以上のRecallを達成**
3. テキストで多くのprobeが必要な原因は「近傍がパーティション境界をまたぐ」ため。これはテキストEmbeddingのランダム-近傍Gapが画像の1/5しかないことに起因
4. **JAはENより少ないprobeで済む**（Gap同等だが、JAの方が類似度分布が広く近傍が見つかりやすい）